## Step 1 – Fetch the raw data from the repository

This step retrieves the raw Facebook comments from the Excel repository, which serve as the input for the subsequent preprocessing.

In [9]:
import os
import json
import pandas as pd

# Folder containing all raw Facebook data stored in CSV files
folder_path = r"C:\Users\ae\OneDrive\Facebook Comments\Batch(6)"
# Output folder for the JSON file
output_path = r"C:\Users\ae\OneDrive\thesis\preprocess"

output = []
current_id = 1

# Loop through every CSV file
for filename in sorted(os.listdir(folder_path)):
    if filename.lower().endswith(".csv"):

        file_path = os.path.join(folder_path, filename)

        try:
            df = pd.read_csv(file_path)

            # Check if the column "Comment Text" is present
            if "Comment Text" not in df.columns:
                print(f"Skipping {filename}: 'Comment Text' column not found.")
                continue

            # Loop through comments
            for comment in df["Comment Text"]:

                if pd.isna(comment):
                    continue

                comment = str(comment).strip()

                if comment == "":
                    continue

                output.append({
                    "id": current_id,
                    "text": comment,
                    "standardized": comment
                })

                current_id += 1

            print(f"Processed {filename}")

        except Exception as e:
            print(f"Error reading {filename}: {e}")

output_file = os.path.join(output_path, "batch_6.json")

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Total comments extracted: {len(output)}")
print(f"JSON file is saved to: {output_file}")

Processed 116.Mopays_143_NTA.csv
Processed 117.Mopays_115_proposals to improve the law.csv
Processed 118.Mopays_81_TRIOLET BUS SERVICE.csv
Processed 119.Mopays_90_Childhood.csv
Processed 120.Mopays_166_La combien points perdi svp.csv
Processed 121.Mopays_224_Shab-E-Baraat.csv
Processed 122.Mopays_357_Help for wedding.csv
Processed 123.Mopays_126_Accident.csv
Processed 124.Mopays_166_Frequentation.csv
Processed 125.TOPFM_150_missing_people.csv
Processed 126.ExpressMaurice_41_Quad seized.csv
Processed 127.TOPNEWS_178_Road_Safety.csv
Processed 128.Mopays_56_autism.csv
Processed 129.Mopays_51_New Eye Hospital in Réduit.csv
Processed 130.Mopays_25_Contracteur.csv
Processed 131.Mopays_224_Machine hit cow.csv
Processed 132.Mopays_146_Arnaqueur.csv
Processed 133.Mopays_-86_items_2026-02-07.csv
Processed 134.Mopays_39_MessagefromSarvesh.csv
Processed 135.PimaInfo_45_Ford.csv
Processed 136.Mopays_306_Mauritius public debt .csv
Processed 137.ExpressMaurice_53_65-year-old northerner suffering from

## Step 2 – Rule-based preprocessing pipeline

This step consists of the following tasks:

1. Apply the three rule-based preprocessing functions in sequence.
2. Apply the preprocessing pipeline on the raw JSON dataset

In [6]:
import re

# --------------------------------------------------------------------
# Apply the three rule-based preprocessing functions in sequence
# --------------------------------------------------------------------

def remove_formatting_artifacts(text):
    """
    Remove formatting artifacts that are not part of the text itself.
    """
    # Replace newlines/tabs with spaces
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = text.replace("\r", " ")

    # Remove escaped quotation marks
    text = text.replace('\\"', "")

    # Remove repeated quotation marks (e.g. """, """")
    text = re.sub(r'"{2,}', "", text)

    # Remove standalone repeated question marks (e.g. ???????)
    text = re.sub(r'\s+\?{2,}(?=\s|$)', " ", text)

    return text

def normalize_text_spacing(text):
    """
    This function normalizes the spacing in a given text by removing
    extra spaces and ensuring that words are separated by a single space.
    Example: Sa oui, progrès   par malediction → Sa oui, progrès par malediction
    """
    return " ".join(text.split())

def normalize_punctuation_spacing(text):
    """
    This function fixes punctuation spacing inconsistencies:
    - Removes spaces before punctuation.
    - Ensures one space after punctuation when followed by a word.
    - Does not add a space if punctuation is at the end.
    """
    # Remove spaces before punctuation
    text = re.sub(r'\s+([.,!?;:])', r'\1', text)

    # Ensure exactly one space after punctuation if followed by a word
    text = re.sub(r'([.,!?;:])\s*([A-Za-zÀ-ÖØ-öø-ÿ])', r'\1 \2', text)

    # Remove multiple spaces
    text = re.sub(r' {2,}', ' ', text)

    return text.strip()

def preprocess(text):
    """
    Preprocessing is performed in the followed order
    """
    text = remove_formatting_artifacts(text)
    text = normalize_text_spacing(text)
    text = normalize_punctuation_spacing(text)
    return text

In [10]:
import json

# --------------------------------------------------
# 1. Set the input and output dataset paths
# --------------------------------------------------
input_path = r"C:\Users\ae\OneDrive\thesis\preprocess\batch_6.json"
output_path = r"C:\Users\ae\OneDrive\thesis\preprocess\batch_6_preprocessed.json"
# input_path = r"C:\Users\ae\OneDrive\thesis\data\merged.json"
# output_path = r"C:\Users\ae\OneDrive\thesis\data\merged.json"
# --------------------------------------------------
# 2. Load the raw/uncleaned dataset
# --------------------------------------------------
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# --------------------------------------------------
# 3. Apply the full rule-based preprocessing pipeline
# --------------------------------------------------
for item in data:
    if "text" in item:
        item["text"] = preprocess(item["text"])

    if "standardized" in item:
        item["standardized"] = preprocess(item["standardized"])

# --------------------------------------------------
# 4. Save the preprocessed dataset
# --------------------------------------------------
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4, ensure_ascii=False)

print("Preprocessing completed")
print("Saved to:", output_path)

Preprocessing completed
Saved to: C:\Users\ae\OneDrive\thesis\preprocess\batch_6_preprocessed.json


## Step 3 - Filter out non-target languages [English]

Language identification is performed using the FastText model available on:

https://fasttext.cc/docs/en/language-identification.html

This step is used to filter out English sentences from the dataset based on the model’s confidence score.

A sentence is flagged if:
- The predicted language is **English (en)**
- The confidence score is **greater than 0.90**

```python
flagged = (language in ["en"]) and (confidence > 0.90)
```

The FastText language identification model is applied on the cleaned/preprocessed dataset to generate language labels, confidence scores, and flagged entries.
```

In [3]:
import json
import pandas as pd
import fasttext

# ----------------------------
# Load FastText model
# ----------------------------
model = fasttext.load_model("lid.176.bin")

# ----------------------------
# Load cleaned/preprocessed JSON file
# ----------------------------
file_path = r"C:\Users\ae\OneDrive\thesis\data\data_versions\mc_joint_text_restoration.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# ----------------------------
# Storage for results
# ----------------------------
results = []

# ----------------------------
# Process each record to determine if the language is English
# ----------------------------
for item in data:
    text = item.get("text", "")
    id_ = item.get("id")

    if not text or not text.strip():
        continue

    # Predict language
    labels, probs = model.predict(text)

    language = labels[0].replace("__label__", "")
    confidence = float(probs[0])

    # Flag logic
    flagged = (language in ["en"]) and (confidence > 0.90)

    results.append({
        "id": id_,
        "text": text,
        "language": language,
        "confidence": confidence,
        "flagged": flagged
    })

# ----------------------------
# Create DataFrame
# ----------------------------
df = pd.DataFrame(results)

# Only flagged rows
flagged_df = df[df["flagged"] == True]

# print(df.head())
print("\nFLAGGED ONLY:\n")
print(flagged_df.head())

# ----------------------------
# Save outputs
# ----------------------------
df.to_csv("language_detection_results.csv", index=False, encoding="utf-8")
flagged_df.to_csv("flagged_only.csv", index=False, encoding="utf-8")


FLAGGED ONLY:

          id                                               text language  \
12706  12707  Bizin boycott this, bane politicians are afrai...       en   
13344  13345  Nou PA ouler ouu also nou bizin dimoune who is...       en   
13584  13585  Listen, it's your wife who says zot pa p under...       en   

       confidence  flagged  
12706    0.911120     True  
13344    0.917827     True  
13584    0.980596     True  


In [14]:
# ----------------------------
# IDs that should NOT be removed
# ----------------------------
keep_flagged_ids = {
    542,
    1044,
    1351,
    1995,
    5472,
    5559,
    8782,
    9063,
    9288,
    14003
}

# ----------------------------
# Remove all flagged IDs EXCEPT the ones above
# ----------------------------
flagged_ids = set(flagged_df["id"]) - keep_flagged_ids

cleaned_data = [
    item for item in data
    if item.get("id") not in flagged_ids
]

# ----------------------------
# Stats
# ----------------------------
original_count = len(data)
removed_count = len(flagged_ids)
remaining_count = len(cleaned_data)

print("\nDATASET STATS")
print("----------------------------")
print(f"Original rows : {original_count}")
print(f"Removed rows  : {removed_count}")
print(f"Remaining rows: {remaining_count}")

# ----------------------------
# Overwrite original file
# ----------------------------
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=4)

print("\nOriginal JSON file cleaned and overwritten successfully.")


DATASET STATS
----------------------------
Original rows : 15085
Removed rows  : 170
Remaining rows: 14915

Original JSON file cleaned and overwritten successfully.


## Step 4 – Identify emoji-only entries

In this step, the dataset is scanned to identify records that contain only emojis, symbols, or whitespace, with no textual content. These records are flagged and counted for manual inspection before being removed from the dataset, as they do not contribute useful linguistic information for the text normalization task.

In [193]:
import json
import re
import pandas as pd

# ----------------------------
# Load dataset
# ----------------------------
file_path = r"C:\Users\ae\OneDrive\thesis\preprocess\batch_5_preprocessed.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# ----------------------------
# Emoji-only detector
# ----------------------------
def is_only_emojis(text):
    if not isinstance(text, str):
        return False

    return re.search(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9]", text) is None

# ----------------------------
# Storage
# ----------------------------
cleaned_data = []
removed_ids = []
emoji_records = []

# ----------------------------
# Process dataset
# ----------------------------
for item in data:
    text = item.get("text", "")
    id_ = item.get("id")

    if is_only_emojis(text):
        emoji_records.append([id_, text])
        removed_ids.append(id_)
        continue

    cleaned_data.append(item)

# ----------------------------
# Emoji DataFrame
# ----------------------------
emoji_df = pd.DataFrame(emoji_records, columns=["id", "text"])

print("\nEMOJI ROWS SAMPLE:")
print(emoji_df.head(20))

# ----------------------------
# Statistics
# ----------------------------
print("\nEMOJI CLEANING STATS")
print("----------------------------")
print(f"Original rows : {len(data)}")
print(f"Removed rows  : {len(removed_ids)}")
print(f"Remaining rows: {len(cleaned_data)}")

# ----------------------------
# Overwrite SAME file
# ----------------------------
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_data, f, ensure_ascii=False, indent=4)

print("\nDataset cleaned and overwritten successfully")


EMOJI ROWS SAMPLE:
      id                                      text
0    979  𝑭𝒊𝒏𝒊 𝒕𝒊𝒓𝒆́ 𝒔𝒂 𝒏𝒐 𝒘𝒂𝒓𝒏𝒊𝒏𝒈 𝒛𝒕 𝒆𝒏𝒓𝒆𝒕𝒂𝒓𝒅🤣🤣🤣🤣
1   5281        𝟦𝟦 𝗋𝗌 𝖼𝗉𝖾 𝗉𝗅 𝗄𝗂 𝗅𝗂 𝗉 𝗄𝗈𝗓𝖾 𝗌𝖺 𝟧𝟣𝗋𝗌.
2   5995                                    🇲🇺 = 🤡
3  11905         𝟦𝟦 𝗋𝗌 𝖼𝗉𝖾 𝗉𝗅 𝗄𝗂 𝗅𝗂 𝗉 𝗄𝗈𝗓𝖾 𝗌𝖺 𝟧𝟣𝗋𝗌

EMOJI CLEANING STATS
----------------------------
Original rows : 14682
Removed rows  : 4
Remaining rows: 14678

Dataset cleaned and overwritten successfully


## Step 5 - Perform a manual corrections

1. Manually remove sentences with users tag a person because the NER cannot find the names in MRU sentences
2. In sentences where we have name, manually we place a tag: [NAME]

## Step 6 – Dataset augmentation with unpunctuated sentences

1. Create an additional version of each sentence by removing all punctuation from the original sentences.
2. The original punctuated sentences are preserved, and the unpunctuated versions are added to the dataset as augmented training samples.

In [169]:
import json
import re
import string

# ----------------------------
# Remove punctuation only
# ----------------------------
translator = str.maketrans("", "", string.punctuation)

def remove_punctuation(text):
    if not isinstance(text, str):
        return text

    # Remove punctuation only (keep emojis)
    text = text.translate(translator)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ----------------------------
# Check whether text contains punctuation
# ----------------------------
def contains_punctuation(text):
    if not isinstance(text, str):
        return False

    return any(ch in string.punctuation for ch in text)


# ----------------------------
# Load dataset
# ----------------------------
input_path = r"C:\Users\ae\OneDrive\thesis\preprocess\batch_5_preprocessed.json"
output_path = r"C:\Users\ae\OneDrive\thesis\preprocess\batch_5_preprocessed.json"

with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)


# ----------------------------
# Create augmented dataset
# ----------------------------
augmented_data = []

next_id = max(item["id"] for item in data) + 1

for item in data:

    # Keep original record
    augmented_data.append(item)

    # Only augment if the ORIGINAL text contains punctuation
    if contains_punctuation(item["text"]):

        new_item = item.copy()
        new_item["id"] = next_id
        next_id += 1

        new_item["text"] = remove_punctuation(item["text"])
        new_item["standardized"] = remove_punctuation(item["standardized"])

        augmented_data.append(new_item)


# ----------------------------
# Save augmented dataset
# ----------------------------
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(augmented_data, f, indent=4, ensure_ascii=False)

print("Augmentation completed")
print(f"Original dataset size : {len(data)}")
print(f"Augmented dataset size: {len(augmented_data)}")
print(f"New samples added     : {len(augmented_data) - len(data)}")
print("Saved to:", output_path)

Augmentation completed
Original dataset size : 2917
Augmented dataset size: 4281
New samples added     : 1364
Saved to: C:\Users\ae\OneDrive\thesis\preprocess\batch_5_preprocessed.json


Resetting the IDS

In [2]:
import json

# ----------------------------
# Load dataset
# ----------------------------
file_path = r"C:\Users\ae\OneDrive\thesis\data\data_versions\mc_joint_text_restoration.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# ----------------------------
# Reset IDs from 1 to N
# ----------------------------
for i, item in enumerate(data, start=1):
    item["id"] = i

# ----------------------------
# Save back to SAME file
# ----------------------------
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"IDs successfully reset. Total rows: {len(data)}")

IDs successfully reset. Total rows: 13830


Text Normalization

In [ ]:
import json
import re

file_path = r"C:\Users\ae\OneDrive\thesis\data\data_versions\mc_joint_text_restoration.json"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# ----------------------------
# Replacement rules - (Only applied to the "standardized" field)
# ----------------------------
replacements = {
    # "banne": "bann",
    # "Banne": "Bann",
    # "p": "pe",
    # "P": "Pe",
    # "maurice": "Maurice",
    # "moris": "Moris",
    # "monter": "monte",
    # "satisfait": "satisfe",
    # "allumer": "alime",
    # "alumer": "alime",
    # "depasser": "depase",
    # "baisser": "bese",
    # "conduire": "kondire",
    # "été": "ete",
    # "arriver": "arrive",
    # "zistoire": "zistwar",
    # "dimoune": "dimunn",
    # "dimounn": "dimunn",
    # "avouer": "avoue",
    # "gagner": "gagne",
    # "trouvé": "truve",
    # "trouve": "truve",
    # "couyon": "kuyon",
    # "pou": "pu",
    # "crier": "kriye",
    # "criye": "kriye",
    # "Viré": "vire",
    # "viré": "vire",
    # "tourné": "tourne",
    # "atan": "atann",
    # "komas": "komans",
    # "partou":"partu",
    # "enbas": "enba",
    # "rever": "reve",
    # "plass": "plas",
    # "discuter": "discute",
    # "prouver": "prouve",
    # "prier": "priye",
    # "fair": "fer",
    # "tusel": "tousel",
    # "prese": "presse",
    # "Accuser":"Accuse",
    # "couyon": "kuyon",
    # "crever": "creve",
    # "rand": "rande",
    # "taper": "tape",
    # "froter": "frote",
    # "montrer": "montre",
    # "suposer": "sipoze",
    # "sipoz": "sipoze",
    # "enkoler": "ankoler",
    # "concerner": "concerne",
    # "pareille": "pareil",
    # "Mete": "Met",
    # "Mette": "Met",
    # "mete": "met",
    # "mette": "met",
    # "bb": "bebe",
    # "enkor": "ankor",
    # "envi": "anvi",
    # "envie": "anvi",
    # "kpav": "kapav",
    # "partout": "partu",
    # "souvant": "souvent",
    # "arrivé": "arrive",
    # "promess": "promesse",
    # "vey": "veye",
    # "My. T": "My.T",
    # "arriéré": "aryere",
    # "saleté": "salete",
    # "trouver":"truve",
    # "preferer": "prefere",
    # "arete": "aret",
    # "arette": "aret",
    # "mai": "me",
    # "tander": "tande",
    # "lavie": "lavi",
    # "Lavie": "Lavi",
    # "kner": "kone",
    # "cont": "kont",
    # "conte" : "kont",
    # "konte": "kont",
    # "kou":"kut",
    # "coute": "kut",
    # "koute": "kut",
    # "gater": "gate",
    # "content": "kontan",
    # "plusieur": "plizier",
    # "operé": "opere",
    # "lom": "zom",
    # "fermé":"ferme",
    # "chere": "cher",
    # "Manger": "Mange",
    # "manger": "mange",
    # "fané": "fane",
    # "letemp": "letan",
    # "partous": "partu",
    # "roulé": "roule",
    # "tirée": "tire",
    # "Pouvi": "Puvi",
    # "pouvi": "puvi",
    # "preter": "prete",
    # "roul": "roule",
    # "installer": "instale",
    # "morder": "morde",
    # "passer": "passe",
    # "manger": "manze",
    # "coumsa": "kumsa",
    # "cumsa": "kumsa",
    # "endans": "endan",
    # "ress": "res",
    # "Ress":"Res",
    # "tester": "teste",
    # "baize": "bez",
    # "lautorite": "otorite",
    # "lecole": "lekol",
    # "Meter": "Met",
    # "meter": "met",
    # "Au moins": "Omoin",
    # "au moin": "omoin",
    # "Au moin": "omoin",
    # "gta": "GTA",
    # "Bizare": "Bizar",
    # "bizare": "bizare",
    # "rentrer": "rantre",
    # "rentre": "rantre",
    # "rente": "rantre",
    # "salete": "salte",
    # "voyager": "voyage",
    # "consulter": "consulte",
    # "mcbeth": "Macbeth",
    # "eshan": "Eshan",
    # "opére":"opere",
    # "Ncc": "NCC",
    # "habituer": "habitue",
    # "penser": "pense",
    # "siposer": "sipoze",
    # "sipoz": "sipoze",
    # "stresser": "stresse",
    # "continué": "continue",
    # "roshi": "Roshi",
    # "koumsa": "kumsa",
    # "ramasser": "ramasse",
    # "jugroo": "Jugroo",
    # "rod": "rode",
    # "japonais": "Japonais",
    # "fié": "fie",
    # "pé": "pe",
    # "MY. T": "MY.T",
    # "securite": "sekirite",
    # "sirtou": "surtout",
    # "sirtu": "surtout",
    # "Sirtou": "Surtout",
    # "Sirtu": "surtout",
    # "presser": "presse",
    # "Defimedia. Info": "Defimedia.info",
    # "exister": "existe",
    # "sauter": "saute",
    # "déroulé": "derule",
    # "signalé": "signale",
    # "passé": "passe",
    # "habiller": "habille",
    # "reposer": "repose",
    # "cou": "kut",
    # "toulezour": "tulezur",
    # "debourser": "debouse",
    # "coup": "kut",
    # "caserne": "Casernes",
    # "mendianer": "mendiane",
    # "refuser": "refiz",
    # "engagé": "engage",
    # "simitiere": "simityer",
    # "simitier": "simityer",
    # "pensé": "pense",
    # "évité": "evite",
    # "elizabeth": "Elizabeth",
    # "alimé": "alime",
    # "changer": "change",
    # "meriter": "merite",
    # "lendrwa": "landrwa",
    # "protege": "protez",
    # "dresser": "dresse",
    # "ran": "rande",
    # "fimer": "fime",
    # "toujour": "tuzur",
    # "defand": "defane",
    # "defan": "defane",
    # "mbc": "MBC",
    # "vitesse": "vitess",
    # "vitess": "vites",
    # "Vitesse": "Vites",
    # "oblizer": "oblize",
    # "debouché": "debouche",
    # "mediclinic": "Medi-Clinic",
    # "stabiliser": "stabilise",
    # "gentile": "gentil",
    # "imaginer": "imazine",
    # "foder": "fode",
    # "DEROULÉ": "DERULE",
    # "gagné": "gagne",
    # "amizer": "amize",
    # "manzer": "manze",
    # "boir": "bwar",
    # "personn": "personne",
    # "aveugler": "aveugle",
    # "legaliser": "legaliz",
    # "legalise": "legaliz",
    # "legaliz": "legalize",
    # "matelas": "matela",
    # "enlever": "enleve",
    # "pinok": "Pinok",
    # "posté": "poste",
    # "excité": "excite",
    # "manker": "manke",
    # "critiquer": "kritik",
    # "critique": "kritik",
    # "kritiker": "kritik",
    # "craze": "kraz",
    # "mangé": "manze",
    # "fermer": "ferme",
    # "lamande": "laman",
    # "marcher": "marche",
    # "soner": "sone",
    # "rentré": "rantre",
    # "amuser": "amuse",
    # "profité": "profite",
    # "marché": "marse",
    # "marche" : "marse",
    # "deranze": "deranz",
    # "montré": "montre",
    # "habillé": "habille",
    # "securité": "sekirite",
    # "vey": "veye",
    # "Mopays. com": "Mopays.com",
    # "mopays. com": "mopays.com",
    # "le temps": "letan",
    # "jusqua": "juska",
    # "jusqu'a": "juska",
    # "realiser": "realiz",
    # "diminuier": "diminye",
    # "lageurre": "lager",
    # "pual": "puale",
    # "Pual": "Puale",
    # "youtube": "YouTube",
    # "Laise": "Laisse",
    # "quan": "kan",
    # "pre- primary": "pre-primary",
    # "coumance": "komans",
    # "re commence": "rekomans",
    # "la drug": "ladrog",
    # "assister": "assiste",
    # "expliquer": "eksplik",
    # "eksplike": "eksplik",
    # "oulé": "oule",
    # "Gps": "GPS",
    # "gps": "GPS",
    # "dehor": "deor",
    # "deors": "deor",
    # "avancé": "avance",
    # "retarder": "retarde",
    # "progresser": "progresse",
    # "rouler": "roule",
    # "monté": "monte",
    # "bollywood": "Bollywood",
    # "emplas": "enplas",
    # "déroule": "derule",
    # "limage": "zimaz",
    # "c'est": "se",
    # "continuer": "continue",
    # "deborder": "deborde",
    # "obligé": "oblige",
    # "vivre": "viv",
    # "gâté":"gate",
    # "C'est": "C'est",
    # "privé": "prive",
    # "héhé": "hehe",
    # "Trouvé": "Truve",
    # "amerder": "amerde",
    # "déclaré": "deklare",
    # "declare": "deklare",
    # "declarer": "deklare",
    # "deklar": "deklare",
    # "content": "kontan",
    # "kontant": "kontan",
    # "rester": "reste",
    # "zoreille": "zorey",
    # "zoreil": "zorey",
    # "sud afrique": "Sud Afrique",
    # "chine": "Chine",
    # "diminuer": "diminye",
    # "diminier": "diminye",
    # "diminue": "diminye",
    # "respecter": "respecte",
    # "pren": "pran",
    # "Pren": "Pran",
    # "intereser": "interese",
    # "gratter": "grate",
    # "Zistoire": "Zistwar",
    # "touché": "tus",
    # "touche": "tus",
    # "tusse": "tus",
    # "tuse":"tus",
    # "d'aprés": "dapre",
    # "bour": "boure",
    # "Bour": "Boure",
    # "desordre": "dezord",
    # "dezorde": "dezord",
    # "nager": "nage",
    # "letage": "letaz",
    # "badein": "Badein",
    # "lider": "lide",
    # "dapré": "dapre",
    # "fole": "folle",
    # "donneé": "donne",
    # "hesité": "hesite",
    # "hesité": "hesite",
    # "discipliner": "discipline",
    # "Al khiz": "Al Khiz",
    # "manier": "maniere",
    # "tardé": "tarde",
    # "Deviner": "Devine",
    # "deviner": "devine",
    # "longtemp": "lontan",
    # "veiller": "veille",
    # "augmenter": "augmente",
    # "deboucher": "debouche",
    # "canibal": "kanibal",
    # "cannibal": "kanibal",
    # "canibale": "kanibal",
    # "areté": "aret",
    # "arete": "aret",
    # "douter": "doute",
    # "azout": "azoute",
    # "ajoute": "azoute",
    # "Ajouté": "Ajoute",
    # "faciliter": "facilite",
    # "zefort": "zefor",
    # "garder": "garde",
    # "Obliger": "Oblize",
    # "oblige": "oblize",
    # "obliger": "oblize",
    # "phar": "phare",
    # "lebras": "lebra",
    # "enbas": "enba",
    # "garder": "garde",
    # "aider": "aide",
    # "Tne": "Tonn",
    # "tne": "tonn",
    # "tasser": "tasse",
    # "contribuer": "kontribye",
    # "contribue": "kontribye",
    # "verifier": "verifie",
    # "serier": "serye",
    # "résulta": "rezilta",
    # "vey": "veye",
    # "Lin": "Linn",
    # "lin": "linn",
    # "baisse": "bese",
    # "fan": "fane",
    # "Channelnews. Mu": "Channelnews.mu",
    # "Retourner": "Retourne",
    # "Defimedia. info": "Defimedia.info",
    # "drogué": "drogue",
    # "possédé": "possede",
    # "interesser": "interese",
    # "interessé": "interese",
    # "menacer": "menace",
    # "arreter": "aret",
    # "atane": "atann",
    # "l'express": "l'Express",
    # "zames": "zame",
    # "Juste": "Just",
    # "verité": "verite",
    # "deceder": "decede",
    # "cassé": "kase",
    # "beneficier": "beneficie",
    # "menacer": "menace",
    # "lévé": "leve",
    # "éna": "ena",
    # "mirail": "miray",
    # "prefer": "prefere",
    # "roder": "rode",
    # "analyse": "analise",
    # "analyser": "analise",
    # "visiter": "visite",
    # "gagnée": "gagne",
    # "juste": "just",
    # "vraimm": "vremem",
    # "connais": "kone",
    # "conzé":"conge",
    # "conze": "conge",
    # "ca": "sa",
    # "Toujour": "Tuzur",
    # "coltar": "coaltar",
    # "koltar": "coaltar",
    # "toulejour": "tulezur",
    # "Toulejour": "tulezur",
    # "recruter":"rekrite",
    # "nommé": "nomme",
    # "costimé": "costime",
    # "toléré": "tolere",
    # "annoncé": "annonce",
    # "concerné": "concerne",
    # "refuser": "refiz",
    # "refuse": "refiz",
    # "badiner": "badine",
    # "amir khan": "Amir Khan",
    # "tuer": "tue",
    # "employée":"employee",
    # "rasser": "rasse",
    # "joanna beranger": "Joanna Beranger",
    # "zazer": "zaze",
    # "plorer": "plore",
    # "TROUVÉ": "TRUVE",
    # "TROUVE": "TRUVE",
    # "beranger": "Beranger",
    # "Hanter": "Hante",
    # "dangerer": "danzere",
    # "danzerer": "danzere",
    # "Danzerer": "Danzere",
    # "çà": "sa",
    # "morisiens": "Morisiens",
    # "deroulé": "derule",
    # "fatiguer": "fatigue",
    # "débordé": "deborde",
    # "Fermer": "Ferme",
    # "Penser": "Pense",
    # "legalizer": "legalize",
    # "legaliz": "legalize",
    # "legaliser": "legalize",
    # "legaliz": "legalize",
    # "rupi":"roupie",
    # "roupi": "roupie",
    # "bizarre": "bizar",
    # "bizare": "bizar",
    # "envoie": "avoy",
    # "tombé": "tombe",
    # "saper": "sape",
    # "la même": "lamem",
    # "prononcer": "prononce",
    # "enn ta": "enta",
    # "ene tas": "enta",
    # "decriminalizer": "decriminalize",
    # "lakoue": "lakour",
    # "neveu": "neve",
    # "newe": "neve",
    # "li même": "limem",
    # "Li même": "limem",
    # "Liem" : "Limem",
    # "defoncer": "defonce",
    # "fatale": "fatal",
    # "saturer": "sature",
    # "vallée pitot": "Vallée Pitot",
    # "serré": "serre",
    # "condamné": "kondann",
    # "kondan":"kondann",
    # "lider": "lide",
    # "l’esprit": "lespri",
    # "lesprit": "lespri",
    # "defan": "defane",
    # "defande": "defane",
    # "possibe": "posib",
    # "possib": "posib",
    # "Possibe": "Posib",
    # "Possib": "Posib",
    # "remark": "remark",
    # "fumé": "fume",
    # "amené" : "amene",
    # "coumencer": "komans",
    # "galouper": "galoupe",
    # "décider": "decide",
    # "meme": "mem",
    # "vraie": "vrai",
    # "la vie": "lavi",
    # "lavie" : "lavi",
    # "demand": "demande",
    # "roche noire": "Roches Noires",
    # "difé": "dife",
    # "honête": "honnête",
    # "Rèy": "Raye",
    # "CONAIT": "KONE",
    # "coulé": "coule",
    # "jugée": "zize",
    # "jugé": "zize",
    # "Rappelle": "Rapel",
    # "MALÈRE": "MALER",
    # "Cadré": "Cadre",
    # "peté": "pete",
    # "frustré": "frustre",
    # "fatiguer": "fatigue",
    # "fatigué": "fatigue",
    # "fatigueé": "fatigue",
    # "oblizé": "oblize",
    # "ARRIVÉ": "ARRIVE",
    # "CHANGÉ": "CHANGE",
    # "ramasseé": "ramasse",
    # "ramassé": "ramasse",
    # "fanner": "fane",
    # "tranquille": "trankil",
    # "tranquil": "trankil",
    # "bambous": "Bambous",
    # "sintetic": "sintetik",
    # "109 mm": "109mm",
    # "Partous": "Partu",
    # # "Partou": "Partu",
    # "Sintetic": "Sintetik",
    # "jumbo": "Jumbo",
    # "parey": "pareil",
    # "pareye": "pareil",
    # "Parey": "Pareil",
    # "enregistrer": "enregistre",
    # "les temps": "letan",
    # "Lapolice": "Lapolis",
    # "préféré": "prefere",
    # "acheter": "achete",
    # "retrouvé": "retrouve",
    # "enbalaho": "enbaloa",
    # "respecté": "respect",
    # "rayé" : "raye",
    # "zomm": "zom",
    # "l'adsu": "ADSU",
    # "l'ADSU": "ADSU",
    # "Fatiguer": "Fatigue",
    # "forcer": "force",
    # "rappelle": "rapel",
    # "besser": "bese",
    # "noyé": "noye",
    # "Coupez": "Coupe",
    # "plaigner": "plaigne",
    # "Sunday times": "Sunday Times",
    # "elever": "eleve",
    # "lapolice": "lapolis",
    # "pezé": "peze",
    # "Sa v dir": "Savedir",
    # "ENVIE": "ANVI",
    # "Trouver": "Truve",
    # "malgache": "Malgache",
    # "conbien": "komie",
    # "Alerment": "Alerman",
    # "alerment": "alerman",
    # "fiencé": "fiancé",
    # "Nlta": "NLTA",
    # "nlta": "NLTA",
    # "Tiou": "Tiew",
    # "Envoye": "Avoy",
    # "compliker": "complike",
    # "bouger": "bouge",
    # "kome": "kone",
    # "proteger": "protez",
    # "pouriture": "pouritir",
    # "tarder": "tarde",
    # "oublier": "oublie",
    # "honter": "honte",
    # "lavé": "lave",
    # "les peuples": "lepep",
    # "zenes": "zeness",
    # "tracé": "trasse",
    # "truver": "truve",
    # "Truver": "Truve",
    # "Lofé": "Lofe",
    # "nide": "lide",
    # "Nide": "Lide",
    # "reculer": "recule",
    # "imaziner": "imazine",
    # "linde": "Inde",
    # "lind": "linde",
    # "Soit disant": "Swadizan",
    # "soit disant": "swadizan",
    # "avancer": "avance",
    # "la kess": "lakess",
    # "Mettre": "Met",
    # "mettre": "met",
    # "roches noires":"Roches Noires",
    # "rappel": "rapel",
    # "rappele": "rapel",
    # "rapele": "rapel",
    # "Rappel" : "Rapel",
    # "Rappele": "Rapel",
    # "Rapele": "Rapel",
    # "lexpress": "l'Express",
    # "donner": "donne",
    # "Faudrer": "Fode",
    # "faudrer": "fode",
    # "royosss": "royoss",
    # "impé": "enpe",
    # "troube": "truve",
    # "Rappelle": "Rapel",
    # "faner": "fane",
    # "insulter": "insulte",
    # "casser": "kase",
    # "lheure": "ler",
    # "fautée": "faute",
    # "fauté": "faute",
    # "constiper": "constipe",
    # "profiter": "profite",
    # "réssi": "resi",
    # "Réssi": "Resi",
    # "rési": "resi",
    # "Rési": "Resi",
    # "invit": "invite",
    # "tous sa": "tusa",
    # "Tou sa": "Tusa",
    # "Tous sa": "Tusa",
    # "esperee": "espere",
    # "babooram": "Babooram",
    # "Rendre": "Rande",
    # "rendre": "rande",
    # "Comic": "Comik",
    # "comic": "comik",
    # "komik": "comik",
    # "Komik": "Comik",
    # "Baisser": "Bese",
    # "frer": "frere",
    # "La vie": "Lavi",
    # "Foss": "Fos",
    # "foss": "fos",
    # "Tous les temp": "Tuletan",
    # "le temp": "letan",
    # "temp": "tan",
    # "Temp" : "Tan",
    # "Le temp": "Letan",
    # "ca": "sa",
    # "covid": "Covid",
    # "l'Inde": "Inde",
    # "panse": "pense",
    # "Panse": "Pense",
    # "mai": "me",
    # "Mai": "Me",
    # "kadrer": "kadre",
    # "kadré": "kadre",
    # "roupies": "roupie",
    # "souillac": "Souillac",
    # "bisin": "bizin",
    # "Bisin": "Bizin",
    # "blier": "bliye",
    # "Blier": "Bliye",
    # "tomber": "tombe",
    # "l'age": "laz",
    # "L'age": "Laz",
    # "age": "laz",
    # "pareye": "pareil",
    # "Adsu": "ADSU",
    # "adsu": "ADSU",
    # "saem": "samem",
    # "Saem": "Samem",
    # "laid": "aide",
    # "Leress": "Leres",
    # "leress": "leress",
    # "Ress": "Res",
    # "ress": "res",
    # "labas": "laba",
    # "la bas": "laba",
    # "Labas": "Laba",
    # "La bas": "Laba",
    # "TOMBER": "TOMBE",
    # "Tomber": "Tombe",
    # "semin": "chemin",
    # "Semin": "Chemin",
    # "tatoné": "tatone",
    # "avoué": "avoue",
    # "avouer": "avoue",
    # "se'est": "se",
    # "beser": "bese",
    # "kozer": "koze",
    # "mover": "move",
    # "tomé": "tombe",
    # "kitte": "kite",
    # "kit": "kite",
    # "diskour": "discour",
    # "eter": "ete",
    # "temps": "tan",
    # "temp": "tan",
    # "racontes": "raconte",
    # "deroulé": "derule",
    # "levé": "leve",
    # "fifa": "FIFA",
    # "rasse": "rase",
    # "esperer": "espere",
    # "floreal": "Floreal",
    # "eclat": "eclate",
    # "ceb": "CEB",
    # "mériter": "merite",
    # "zenfans": "zanfan",
    # "considerer": "considere",
    # "fernal": "fermal",
    # "la tete": "latet",
    # "oublié": "oublie",
    # "Oublié": "Oublie",
    # "oublier": "oublie",
    # "2sieme": "2eme",
    # "commencer": "commence",
    # "popol": "Popol",
    # "couraze": "kouraz",
    # "couraz": "kouraz",
    # "resultat": "rezilta",
    # "lipie": "lipied",
    # "Pareille": "Pareil",
    # "pareille": "pareil",
    # "accepter": "accepte",
    # "accepté": "accepte",
    # "Accepter": "Accepte",
    # "Accepté": "Accepte",
    # "Imbecile": "Inbesil",
    # "imbecile": "inbesil",
    # "kriy" :"kriye",
    # "Kriy":"Kriye",
    # "enplace": "enplas",
    # "constater": "constate",
    # "MONTER": "MONTE",
    # "pisser": "pisse",
    # "dès": "des",
    # "Baisser": "Bese",
    # "pourri": "pouri",
    # "George owell": "George Owell",
    # "demander": "demande",
    # "vit": "vite",
    # "Vit": "Vite",
    # "konn": "kone",
    # "Konn": "Kone",
    # "bousse": "bouss",
    # "Bousse": "Bousse",
    # "sutirer": "sutire",
    # "soutirez": "sutire",
    # "COZE": "KOZE",
    # "vender": "vande",
    # "là": "la",
    # "koz": "koze",
    # "Kozer": "koze",
    # "voter": "vote",
    # "Voter": "Vote",
    # "tou": "tu",
    # "Tou": "Tu",
    # "vinne": "vinn",
    # "bane": "bann",
    # "vine": "vinn",
    # "apres": "apre",
    # "Apres": "Apre",
    # "énan": "ena",
    # "moricien": "Morisien",
    # "dévan": "divan",
    # "nou": "nu",
    # "Nou": "Nu",
    # "pou": "pu",
    # "Pou": "Pu",
    # "narien": "nanyin",
    # "nipah": "Nipah",
    # "aprés": "apre",
    # "ce": "se",
    # "dimoun": "dimunn",
    # "Dimoun": "Dimunn",
    # "zafair": "zafer",
    # "parait": "paret",
    # "in gagn": "inn gagne",
    # "Cki": "Seki",
    # "ine": "inn",
    # "l'inde": "Inde",
    # "pliss": "plis",
    # "fk" : "fek", 
    # "srti": "sorti",
    # "Ine": "Inn",
    # "kine": "kinn",
    # "touille":"touy",
    # "coner": "kuma",
    # "ene": "enn",
    # "couma" : "kuma",
    # "mett": "met",
    # "mm": "mem",
    # "al": "ale",
    # "zotte": "zot",
    # "Zotte": "Zot",
    # "Couillonade": "Kuyonad",
    # "arrete": "aret",
    # "Arrete": "Aret",
    # "ile": "lil",
    # "ressi": "resi",
    # "Bizain": "Bizin",
    # "arrêt": "aret",
    # "kalité": "kalite",
    # "vin": "vinn",
    # "morice": "Moris",
    # "kon": "kone",
    # "In": "Inn",
    # "rodé": "rode",
    # "in": "inn",
    # "ggner":"gagne",
    # "reussi": "resi",
    # "fr": "fer",
    # "lacide": "lasid",
    # "lacid": "lasid",
    # "kout": "kut",
    # "ggn": "gagne",
    # "baté": "bate",
    # "publié": "publie",
    # "circulé": "circule",
    # "c": "se",
    # "confirmè": "confirme",
    # "selmen": "selma",
    # "rol": "role",
    # "bzn": "bizin",
    # "encors": "ankor",
    # "coté": "kote",
    # "coné": "kone",
    # "line":"linn",
    # "Linn": "Linn",
    # "lotoriter": "otorite",
    # "zott": "zot",
    # "fine": "finn",
    # "coum sa": "kumsa",
    # "guet": "get",
    # "l’Inde": "Inde",
    # # "commier": "komie",
    # "conner": "kone",
    # "sah": "sa",
    # "Battez": "Bate",
    # "battez": "bate",
    # "cne": "kone",
    # "Bzn": "Bizin",
    # "bzn" : "bizin",
    # "daccord": "dakor",
    # "pencor": "pankor",
    # "dimounes" : "dimunn",
    # "bannes": "bann",
    # "kt": "kot",
    # "contan": "kontan",
    # "travaille": "travail",
    # "travay": "travail",
    # "bucu": "buku",
    # "bcp": "buku",
    # "Astr": "Aster",
    # "gagn": "gagne",
    # "mne": "monn",
    # "1 ta": "enta",
    # "bat": "bate",
    # "zenfants": "zanfan",
    # "roles": "role",
    # "zte": "zot",
    # "bezé": "beze",
    # "capave": "kapav",
    # "touiye": "touy",
    # "baT": "bate",
    # "merité": "merite",
    # "konner": "kone",
    # "dn": "dan",
    # "meriT": "merite",
    # "samm": "samem",
    # "kumsamm": "kumsamem",
    # "Fr": "Fer",
    # "zenfan": "zanfan",
    # "avk": "avek",
    # "ban": "bann",
    # "touye": "touy",
    # "Kifr": "Kifer",
    # "fodé": "fode",
    # "brulé": "brule",
    # "Lofai": "Lofe",
    # "bater": "bate",
    # "gne": "gagne",
    # "travaillere": "travayer",
    # "travailler": "travayer",
    # "amener": "amene",
    # "depuie": "depi",
    # "merit": "merite",
    # "sauver": "sove",
    # "finne": "finn",
    # "Libeter": "Liberte",
    # "cuma": "kuma",
    # "coumadir": "kumadir",
    # "kifaire": "kifer",
    # "Kifaire": "Kifer",
    # "associê": "associe",
    # "r": "ar",
    # "encr": "ankor",
    # "Cumsa mem": "Kumsamem",
    # "Coumsa mem": "Kumsamem",
    # "brilé": "brile",
    # "lêçon": "lesson",
    # "sanela": "sanla",
    # "dimune": "dimunn",
    # "truv": "truve",
    # "nanier": "nanyin",
    # "koT": "kote",
    # "zenfant": "zanfan",
    # "Maziner": "Mazine",
    # "Koumsa mem": "Kumsamem",
    # "casse": "kase",
    # "kumsa mem": "kumsamem",
    # "kifr": "kifer",
    # "faudé": "fode",
    # "zamais": "zame",
    # "oune": "ou",
    # "cass": "kase",
    # "Kumsa meme": "Kumsamem",
    # "inpé": "enpe",
    # "largent": "larzan",
    # "ferm": "ferme",
    # "Ferm": "Ferme",
    # "comence": "komans",
    # "zafaire": "zafer",
    # "Zafaire": "Zafer",
    # "rent": "rantre",
    # "govt": "guvernman",
    # "croir": "krwar",
    # "bnela": "banla",
    # "casiet": "kasyet",
    # "persone": "personne",
    # "tjur": "tuzur",
    # "cnner": "kone",
    # "atand": "atann",
    # "ggne": "gagne",
    # "pareill": "pareil",
    # "kouma": "kuma",
    # "coser": "koz",
    # "tousa": "tusa",
    # "permit a point": "permis a point",
    # "permi a poin": "permis a point",
    # "conger": "conge",
    # "aler": "ale",
    # "malgres": "malgre",
    # "laterre": "later",
    # "vni": "vini",
    # "couler": "koule",
    # "Isi": "ici",
    # "isi": "ici",
    # "tout lezour": "tulezur",
    # "rose hill": "Rose Hill",
    # "vacoas": "Vacoas",
    # "phoenix": "Phoenix",
    # "4bornes": "Quatre Bornes",
    # "mauritius": "Mauritius",
    # "zt": "zot",
    # "Zt": "Zot",
    # "bondié": "Bondye",
    # "bondier": "Bondye",
    # "bondieu": "Bondye",
    # "Mone": "Mon",
    # "kumen": "kuma",
    # "enn tas": "enta",
    # "fréquenté": "frekante",
    # "lajoue": "lazou",
    # "drmi": "dormi",
    # "amene": "amenn",
    # "Amene": "Amenn",
    # "plitot": "plito",
    # "pna": "pena",
    # "saP": "sape",
    # "tomB": "tombe",
    # "trasser": "trasse",
    # "dal": "dhall",
    # "lacaz": "lakaz",
    # "lcz": "lakaz",
    # "Lacaz": "Lakaz",
    # "qi": "ki",
    # "laba mem": "labamem",
    # "ramgoolam": "Ramgoolam",
    # "venD": "vande",
    # "koner": "kone",
    # "Komiee": "Komie",
    # "res t": "reste",
    # "trover": "truve",
    # "Astere":"Aster",
    # "t": "to",
    # "gratuis": "gratuit",
    # "bondie": "Bondye",
    # "encor": "ankor",
    # "lecol": "lekol",
    # "kpv": "kapav",
    # "trav": "travail",
    # "trava": "travail",
    # "pane": "pann",
    # "lizier": "lizye",
    # "lev": "leve",
    # "barage": "baraz",
    # "circuler": "circule",
    # "chanzer": "chanze",
    # "pan": "pann",
    # "reusi": "resi",
    # "bobone": "bobon",
    # "to mem": "tomem",
    # "cki": "seki",
    # "Cki": "Seki",
    # "Ofe": "Ofet",
    # "det": "dette",
    # "appell": "apel",
    # "mé": "me",
    # "nimporte": "nerport",
    # "crwar": "krwar",
    # "sauV": "sove",
    # "traP": "trape",
    # "cpv": "kapav",
    # "Line": "Linn",
    # "gayne": "gagne",
    # "krk": "korek",
    # "Good land": "Goodlands",
    # "koter": "kote",
    # "deroule": "derule",
    # "enkr": "ankor",
    # "bizn": "bizin",
    # "batt": "bate",
    # "lah": "la",
    # "kumsa même": "kumsamem",
    # "batter": "bate",
    # "manz": "manze",
    # "m": "mo",
    # "cmprn": "konpran",
    # "liyeux": "lizye",
    # "coT": "kote",
    # "dejais": "deza",
    # "bne": "bann",
    # "kne": "kone",
    # "Cpv": "Kapav",
    # "Faude": "Fode",
    # "compren": "konpran",
    # "kma": "kuma",
    # "eT": "ete",
    # "laplie": "lapli",
    # "Lapli": "Lapli",
    # "laveil": "la veille",
    # "déja": "deza",
    # "conGé": "conge",
    # "Ecout": "Ekut",
    # "mpe": "mo pe",
    # "kmprnd": "compran",
    # "Bnla": "Banla",
    # "donn": "donne",
    # "sa mem": "samem",
    # "Akz": "Akoz",
    # "zafr": "zafer",
    # "dimun": "dimunn",
    # "alle": "ale",
    # "bon dieu": "Bondye",
    # "ecut": "ekut",
    # "emba": "enba",
    # "france": "France",
    # "geté": "gete",
    # "veyer": "veye",
    # "zanfant": "zanfan",
    # "mo mm": "momem",
    # "Mo mm": "Momem",
    # "mo mem": "momem",
    # "parsqui": "parski",
    # "KieT": "Kiete",
    # "Dimé": "Dime",
    # "soleille": "soleil",
    # "ereur": "erer",
    # "navin": "Navin",
    # "Rodrig": "Rodrigues",
    # "paC": "passe",
    # "Ceki": "Seki",
    # "dmai": "demain",
    # "la cour": "lakour",
    # "apranne": "aprann",
    # "mone": "monn",
    # "assizé": "asize",
    # "cone": "kone",
    # "Malere": "Maler",
    # "coumanse": "komans",
    # "trist": "triste",
    # "Ene": "Enn",
    # "la mem": "lamem",
    # "mone": "monn",
    # "Mone": "Monn",
    # "deja": "deza",
    # "Bondié": "Bondye",
    # "poster": "poste",
    # "ki faire": "kifer",
    # "Ki faire": "Kifer",
    # "Personnes": "Personne",
    # "personnes": "personne",
    # "capav": "kapav",
    # "allé": "ale",
    # "Allé": "Ale",
    # "Lor la": "Lorla",
    # "lor la": "lorla",
    # "ilegal": "illegal",
    # "tapper": "tape",
    # "kuma dir": "kumadir",
    # "couma dir": "kumadir",
    # "ousois": "ouswa",
    # "juska": "ziska",
    # "Juska": "Ziska",
    # "derouler": "derule",
    # "conne": "kone",
    # "vraimm": "vremem",
    # "gette": "get",
    # "cid": "CID",
    # "kin": "kinn",
    # "Sa même": "Samem",
    # "sa même": "samem",
    # "volere": "voler",
    # "dominere": "dominer",
    # "qud": "kan",
    # "parcequi": "parski",
    # "le pep": "lepep",
    # "vré": "vre",
    # "kisenla": "kisanla",
    # "coz": "koz",
    # "siposé": "sipoze",
    # "Kisenla": "Kisanla",
    # "zimages": "zimage",
    # "kuman": "kuma",
    # "largents": "larzan",
    # "jis": "zis",
    # "jordi": "zordi",
    # "tujur": "tuzur",
    # "bnla": "banla",
    # "Zott": "Zot",
    # "pistass": "pistas",
    # "ousi": "osi",
    # "guete": "get",
    # "cav": "kapav",
    # "don": "donne",
    # "pour vue": "pourvi",
    # ###### NEWLY ADDED
    # "trase": "trasse",
    # "nouvo": "nuvo",
    # "Nouvo": "Nuvo",
    # "trase": "trasse",
    # "Trase": "Trase",
    # "traC": "trasse",
    # "maurice": "Maurice",
    # "Dapres": "Dapre",
    # "tjr": "tuzur",
    # "Tjr": "Tuzur",
    # "retrouver": "retrouve",
    # "retruve": "retrouve",
    # "Retrouver": "Retrouve",
    # "Retruve": "Retrouve",
    # "fess": "fesse",
    # "vreman": "vremem",
    # "depase": "depasse",
    # "courber": "courbe",
    # "progresser": "progresse",
    # "dekuyoner": "dekuyone",
    # "atan": "atann",
    # "atane": "atann",
    # "st hubert": "St Hubert",
    # "rose belle": "Rose Belle",
    # "rever": "reve",
    # "éna": "ena",
    # "jis": "ziss",
    # "necessaire": "neseser",
    # "prouver": "prouve",
    # "papannn": "pann",
    # "Latere": "Later",
    # "latere": "later",
    # "lamane": "laman",
    # "tase": "tasse",
    # "resultats": "rezilta",
    # "tra" : "travail", 
    # "miraille": "miray",
    # "pares": "paress",
    # "zenes": "zeness",
    # "zenese": "zeness",
    # "jeuness": "zeness",
    # "jeness": "zeness",
    # "crazer": "kraze",
    # "kraz": "kraze",
    # "Bondye": "Bondieu",
    # "ramase": "ramasse",
    # "ramas": "ramasse",
    # "paye": "peye",
    # "plusieur": "plizier",
    # "faudé": "fode",
    # "Faudé": "Fode",
    # "PERSONN":"PERSONNE",
    # "personn": "personne",
    # "separer": "separe",
    # "bloker": "bloke",
    # "tou letemps": "tultan",
    # "tuletan": "tultan",
    # "Tou letemps": "Tultan",
    # "aksepte": "accepte",
    # "acepte": "accepte",
    # "demars": "demarse",
    # "amerd": "amerde",
    # "continuer": "kontiyn",
    # "ramass": "ramasse",
    # "laba mem": "labamem",
    # "dimande": "demande",
    # "demander": "demande",
    # "gagner": "gagne",
    # "morice": "Moris",
    # "creol": "kreole",
    # "laire": "lair",
    # "renouveler": "renouvele",
    # "bourer": "boure",
    # "IMPLEMENTER": "IMPLEMENTE",
    # "sanz": "sanze",
    # "zenes": "zeness",
    # "kash": "cash",
    # "rod": "rode",
    # "securite": "sekirite",
    # "devan": "divan",
    # "Devan": "Divan",
    # "seryer": "serye",
    # "assize": "asize",
    # "lordre": "lord",
    # "lorde": "lord",
    # "blok": "bloke",
    # "continue": "kontiyn",
    # "gete": "get",
    # "dessan": "desan",
    # "mone": "mon",
    # "monn": "mon",
    # "lorage": "loraz",
    # "esperé": "espere",
    # "Fatigueé": "Fatigue",
    # "omwin": "omoin",
    # "families": "fami",
    # "zes": "zess",
    # "disan": "disang",
    # "zouer": "zwe",
    # "zoue": "zwe",
    # "delo": "dilo",
    # "Tous": "Tu",
    # "Tout": "Tu",
    # "trembler": "tremble",
    # "priver": "prive",
    # "coz": "koze",
    # "koster": "koste",
    # "coster": "koste",
    # "la cours": "lakour",
    # "Allez": "Ale",
    # "gard": "garde",
    # "Gard": "Garde",
    # "affecter": "affecte",
    # "assiser":"asize",
    # "assize": "asize",
    # "cocain": "kokin",
    # "tu les jours": "tulezur",
    # "tout les jours": "tulezur",
    # "tout les jour": "tulezur",
    # "Cki": "Seki",
    # "achete": "aste",
    # "acheter": "aste",
    # "ACHETER": "ASTE",
    # "Acheter": "Aste",
    # "respirer": "respire",
    # "lagere": "lager",
    # "sss": "SSS",
    # "l'afrique": "l'Afrique",
    # "plusiere": "plizier",
    # "konei": "kone",
    # "enkor": "ankor",
    # "conne": "kone",
    # "côté": "kote",
    # "cigarete": "cigaret",
    # "tirer": "tire",
    # "Ladans": "Ladan",
    # "posé": "pose",
    # "dakr": "dakor",
    # "retourné": "retourne",
    # "Rouler": "Roule",
    # "delo": "dilo",
    # "anil": "Anil",
    # "raconter": "raconte",
    # "deranger": "deranz",
    # "derange": "deranz",
    # "mitd": "MITD",
    # "Payer": "Peye",
    # "paye": "peye",
    # "Paye": "Peye",
    # "payer": "peye",
    # "Ladrogue": "Ladrog",
    # "explik": "eksplik",
    # "Explik": "Eksplik",
    # "Respecter": "Respecte",
    # "frmal": "fermal",
    # "decorer": "decore",
    # "roz hill": "Rose Hill",
    # "bazar" : "bazaar",
    # "taxer": "taxe",
    # "conait": "kone",
    # "trappe": "trape",
    # "trap": "trape",
    # "traper": "trape",
    # "tout": "tu",
    # "Tout": "Tu",
    # "Tou": "Tu",
    # "Tous": "Tu",
    # "tarif": "tariff",
    # "Tarif": "Tariff",
    # "la vie": "lavi",
    # "la peche": "lapess",
    # "nagé": "naze",
    # "nage": "naze",
    # "Retir": "Retire",
    # "retir": "retire"
    # "aid" :"aide",
    # "critike" : "kritik",
    # "la-bas": "là-bas",
    # "ça": "sa",
    # "Ça": "Sa",
    # "Tu ça": "Tusa",
    # "tu ça": "tusa",
    # "lever": "leve",
    # "lamann": "laman",
    # "zaper": "zape",
    # "lalliance": "lalyans",
    # "laliance": "lalyans",
    # "Nord": "nord",
    # "mange": "manze",
    # "Mange": "Manze",
    # "passé": "passe",
    # "Continue": "Kontiyn",
    # "re monte": "remonte",
    # "partout": "partu",
    # "partou": "partu",
    # "Partou": "Partou",
    # "importan": "important",
    # "inportan": "important",
    # "latete": "latet",
    # "appel": "apel",
    # "Appel": "Apel",
    # "Recrute": "Rekrite",
    # "recrute": "rekrite",
    # "permit": "permis",
    # "Permit": "Permis",
    # "baissé": "bese",
    # "caca": "kaka",
    # "cot": "kot",
    # "Encor": "Ankor",
    # "reponce": "reponse",
    # "ranger": "range",
    # "zurer": "zoure",
    # "laver": "lave",
    # "Éna": "Ena",
    # "mème": "mem",
    # "foi": "fwa",
    # "la em": "lamem",
    # "bku": "buku",
    # "daccord": "dakor",
    # "Daccord": "Dakor",
    # "di sang": "disang",
    # "disan": "disang",
    # "vacin": "vaccine",
    # "vaccin": "vaccine",
    # "Fine": "Finn",
    # "cout": "kut",
    # "comico": "komiko",
    # "casern": "Casernes",
    # "ANCOR": "ANKOR",
    # "figir": "figure",
    # "Figir": "Figure",
    # "saken": "sakenn",
    # "inpe": "enpe",
    # "Inpe": "Enpe",
    # "Kumsa mem": "Kumsamem",
    # "fumer": "fume",
    # "peze": "pez",
    # "Pez": "Pez",
    # "pandi": "pendi",
    # "Esperon": "Esperons",
    # "esperon": "Esperon",
    # "debarasser": "debarasse",
    # "moricien": "Morisien",
    # "zenes": "zeness",
    # "Zenes": "Zeness",
    # "swa": "soi",
    # "Swa": "Soi",
    # "sanzer": "sanze",
    # "Telman": "Telma",
    # "telman": "telma",
    # "Alerma": "Alerman",
    # "alerma": "alerman",
    # "Fumée": "Fume",
    # "la caz": "lakaz"
    }

# ----------------------------
# Apply replacements
# ----------------------------
total_changes = 0

for item in data:
    text = item.get("standardized", "")

    for old, new in replacements.items():
        # Replace whole words only
        pattern = r"\b" + re.escape(old) + r"\b"
        text, count = re.subn(pattern, new, text)
        total_changes += count

    item["standardized"] = text

# ----------------------------
# Save back to same file
# ----------------------------
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print(f"Done! Total replacements made: {total_changes}")

Done! Total replacements made: 80
